### Imports

In [70]:
import pandas as pd
import numpy as np
import fastf1 as fastf1
fastf1.Cache.enable_cache("../cache")
import fastf1.ergast as ergast
import datetime
import json
import time
import optuna
import matplotlib.pyplot as plt
import plotly.io as show


from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

from xgboost import XGBRanker

### 1. Basic Cleaning and splitting the dataset
#### 1.1 Reading in the Data and Inspection

In [71]:
raw_data = pd.read_csv("../data/driver_results_final.csv")


In [72]:
pd.set_option("display.max_columns", None)
raw_data.head()

,Round,Year,RaceName,MeetingKey,SessionKey,CircuitKey,CircuitShortName,StartDate,DriverNumber,FullName,Abbreviation,DriverId,TeamId,Position,GridPosition,ClassifiedPosition,Status,Points,Time,Laps,AirTemp,RainFall,WindSpeed,QualiTime,DriverPoints,ConstructorPoints,TeamAverageFinishEWMA,EWMAFinishPosition,relevance_score
0,1,2018,Australian Grand Prix,979,5066,10,Melbourne,2018-03-25 16:10:00,5,Sebastian Vettel,VET,vettel,ferrari,1.0,3.0,1,Finished,25.0,0 days 01:29:33.283000,58.0,24.077477,0.045045,3.691892,0 days 00:00:00.674000,25.0,40.0,0.0,0.0,22.0
1,1,2018,Australian Grand Prix,979,5066,10,Melbourne,2018-03-25 16:10:00,44,Lewis Hamilton,HAM,hamilton,mercedes,2.0,1.0,2,Finished,18.0,0 days 00:00:05.036000,58.0,24.077477,0.045045,3.691892,0 days 00:00:00,18.0,22.0,0.0,0.0,21.0
2,1,2018,Australian Grand Prix,979,5066,10,Melbourne,2018-03-25 16:10:00,7,Kimi Räikkönen,RAI,raikkonen,ferrari,3.0,2.0,3,Finished,15.0,0 days 00:00:06.309000,58.0,24.077477,0.045045,3.691892,0 days 00:00:00.664000,15.0,40.0,0.0,0.0,20.0
3,1,2018,Australian Grand Prix,979,5066,10,Melbourne,2018-03-25 16:10:00,3,Daniel Ricciardo,RIC,ricciardo,red_bull,4.0,8.0,4,Finished,12.0,0 days 00:00:07.069000,58.0,24.077477,0.045045,3.691892,0 days 00:00:00.988000,12.0,20.0,0.0,0.0,19.0
4,1,2018,Australian Grand Prix,979,5066,10,Melbourne,2018-03-25 16:10:00,14,Fernando Alonso,ALO,alonso,mclaren,5.0,10.0,5,Finished,10.0,0 days 00:00:27.886000,58.0,24.077477,0.045045,3.691892,0 days 00:00:02.433000,10.0,12.0,0.0,0.0,18.0


In [73]:
# Check for NaN / missing values
nan_mask = raw_data.isna()
print(f"Total NaNs in y: {nan_mask.sum()}")

Total NaNs in y: Round                    0
Year                     0
RaceName                 0
MeetingKey               0
SessionKey               0
CircuitKey               0
CircuitShortName         0
StartDate                0
DriverNumber             0
FullName                 0
Abbreviation             0
DriverId                 0
TeamId                   0
Position                 0
GridPosition             3
ClassifiedPosition       0
Status                   0
Points                   0
Time                     0
Laps                     0
AirTemp                  0
RainFall                 0
WindSpeed                0
QualiTime                0
DriverPoints             0
ConstructorPoints        0
TeamAverageFinishEWMA    0
EWMAFinishPosition       0
relevance_score          0
dtype: int64


#### 1.2 Data cleaning

In [74]:
data_df = raw_data.copy()

In [75]:
col_name = {
    'Round':"round",
    'Year':"year",
    'RaceName':"race_name",
    'MeetingKey':"meeting_key",
    'SessionKey':"session_key",
    'CircuitKey':"circuit_key",
    'CircuitShortName':"circuit_name",
    'StartDate':"start_date",
    'DriverNumber':"driver_number",
    'FullName':"full_name",
    'Abbreviation':"abv",
    'DriverId':"driver_id",
    'TeamId':"team_id",
    'Position':"finish_position",
    'GridPosition':"grid_position",
    'ClassifiedPosition':"classified_position",
    'Status':"status",
    'Points':"race_points",
    'Time':"race_finish_time",
    'Laps':"laps",
    'AirTemp':"air_temp",
    'RainFall':"rainfall",
    'WindSpeed':"wind_speed",
    'QualiTime':"quali_relative_time",
    'DriverPoints':"driver_points",
    'ConstructorPoints':"constructor_points",
    'TeamAverageFinishEWMA':"constructors_ewma",
    'EWMAFinishPosition':"driver_ewma",
    'relevance_score':"relevance_score"
}

data_df.rename(columns=col_name, inplace=True)

In [76]:
data_df.drop(['start_date','status','full_name','abv','race_points','meeting_key','circuit_name','circuit_key','round','race_name','driver_number','classified_position','finish_position','grid_position','laps','team_id','race_finish_time','driver_points', 'constructor_points'],axis=1,inplace=True)
data_df = data_df.sort_values(by=['year', 'session_key']).reset_index(drop=True)

In [77]:
data_df['quali_relative_time'] = pd.to_timedelta(data_df['quali_relative_time'])
#kind-off data scaling and encoding
data_df['quali_relative_time'] = data_df['quali_relative_time'].map(lambda x : ((x / datetime.timedelta(microseconds=1))/1000))

In [78]:
for col in ['constructors_ewma', 'driver_ewma']:
    data_df[col] = data_df[col].map(lambda x : 20.0 if pd.isna(x) else x)

In [79]:
for col in ['relevance_score']:
    data_df[col] = data_df[col].map(lambda x : 0.0 if pd.isna(x) else x)

#### 1.3 Splitting the Data

In [80]:
cleaned_df = data_df.copy()

In [81]:
#split and sort by session key after sort
train_mask = cleaned_df['year'] < 2024
test_mask = cleaned_df['year'] >= 2024

train_df = cleaned_df[train_mask]
test_df = cleaned_df[test_mask]


In [82]:
#sort data by session key
train_df = train_df.sort_values(by=['session_key'])
test_df = test_df.sort_values(by=['session_key'])

sorted_df = cleaned_df.sort_values(by=['session_key'])

In [83]:
y_train = train_df['relevance_score'].copy()
y_test = test_df['relevance_score'].copy()
y = sorted_df['relevance_score'].copy()
y.head()


0     22.0
19     3.0
18     4.0
17     5.0
16     6.0
Name: relevance_score, dtype: float64

In [84]:
X_train = train_df.copy()
X_train.drop(['relevance_score','year','session_key','driver_id'], axis=1, inplace=True)
X_train.head()

X_test = test_df.copy()
X = sorted_df.copy()
X.drop(['relevance_score','year','session_key', 'driver_id'], axis=1, inplace=True)
X.head()

one_session_mask = X_test['session_key'] == 11361


X_test.drop(['relevance_score','year','session_key','driver_id'], axis=1, inplace=True)
X_test.head()

,air_temp,rainfall,wind_speed,quali_relative_time,constructors_ewma,driver_ewma
2500,18.227389,0.0,0.785987,14.0,5.645641,1.007152
2519,18.227389,0.0,0.785987,1605.0,29.041414,15.350103
2518,18.227389,0.0,0.785987,1591.0,0.000000,17.732445
2517,18.227389,0.0,0.785987,1783.0,21.378545,11.578794
2516,18.227389,0.0,0.785987,1628.0,21.378545,9.799751


In [85]:
#extract the qid
qid_train = train_df.session_key.copy()

qid_test = test_df.session_key.copy()

qid = sorted_df.session_key.copy()
qid.head()

qid_train.head()

0     5066
19    5066
18    5066
17    5066
16    5066
Name: session_key, dtype: int64

In [86]:
print("--- TARGET DATA CHECK ---")
print("y_train min/max:", y_train.min(), y_train.max())
print("y_train NaNs:", y_train.isna().sum())

print("\n--- FEATURE DATA TYPES ---")
print(X_train.dtypes)

print("\n--- GROUP SIZE CHECK ---")
qid_series = pd.Series(qid_train)
print("Unique sessions:", qid_series.nunique())
print("Drivers per session (min/mean/max):", 
      qid_series.value_counts().min(),
      qid_series.value_counts().mean(),
      qid_series.value_counts().max())

--- TARGET DATA CHECK ---
y_train min/max: 3.0 22.0
y_train NaNs: 0

--- FEATURE DATA TYPES ---
air_temp               float64
rainfall               float64
wind_speed             float64
quali_relative_time    float64
constructors_ewma      float64
driver_ewma            float64
dtype: object

--- GROUP SIZE CHECK ---
Unique sessions: 125
Drivers per session (min/mean/max): 20 20.0 20


In [87]:
#Hyper-parameter tuning
# param_grid = {
#     'n_estimators' : [50,100,150,200,270],
#     'learning_rate' : [0.005,0.01,0.1,0.2],
#     'max_depth' : [3,5,7],
#     'max_leaves' : [5,7,10],
#     'grow_policy' : ['depthwise', 'lossguide'],
#     'min_child_weight': [1,2,3,5],
#     'subsample' : [0.65, 0.7, 0.8, 1.0],
#     'gamma' : [0.0, 0.1, 0.5, 1.0, 3.0],
#     'reg-alpha' : [0, 0.1, 1, 10, 100],
#     'reg_lambda' : [0.0,0.1,1.0,1.5],
#     'lambdarank_num_pair_per_sample' : [5,10,20] #needed?
# }

# ranker = XGBRanker(tree_method='exact', ndgc_exp_gain=False, lambdarank_unbiased=False, lambdarank_pair_method='mean', eval_metric='ndgc',  objective="rank:ndcg")

# gkf = GroupKFold(n_splits=10)

# grid_search = RandomizedSearchCV(estimator=ranker, param_distributions=param_grid, n_iter=150, n_jobs=-1, cv=gkf, verbose=2, random_state=63, scoring='accuracy')
# grid_search.fit(X_train, y_train,)

# print("best param :", grid_search.best_params_)
# print("best accuracy :", grid_search.best_score_)


In [88]:
#using optuna to hyperparameterize
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    param_grid = {
    'n_estimators' : trial.suggest_int('n_estimators',50,500),
    'learning_rate': trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
    'max_depth' : trial.suggest_int('max_depth',3,10),
    'max_leaves' : trial.suggest_int('max_leaves',5,30),
    "objective": "rank:ndcg",
    "eval_metric": "ndcg",
    'early_stopping_rounds' : 30,
    "tree_method": "exact",
    "ndcg_exp_gain" : False,
    "lambdarank_unbiased" : False,
    "lambdarank_pair_method" : 'mean',
    "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
    "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    'min_child_weight': trial.suggest_int('min_child_weight', 1, 6),
    "gamma": trial.suggest_float("gamma", 0.0, 3.0)
    #'lambdarank_num_pair_per_sample' : [5,10,20] #needed?
    }

    gkf = GroupKFold(n_splits=5)
    cv_scores = []

    for train_idx, val_idx in gkf.split(X, y, groups=qid):
        model = XGBRanker(**param_grid)
        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
            qid=qid.iloc[train_idx],
            eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
            eval_qid=[qid.iloc[val_idx]],
            verbose=2,
        )
        
        cv_scores.append(model.best_score)

    return np.mean(cv_scores)


In [89]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[0]	validation_0-ndcg:0.86319
[2]	validation_0-ndcg:0.92678
[4]	validation_0-ndcg:0.93500
[6]	validation_0-ndcg:0.93931
[8]	validation_0-ndcg:0.93916
[10]	validation_0-ndcg:0.94125
[12]	validation_0-ndcg:0.94216
[14]	validation_0-ndcg:0.94257
[16]	validation_0-ndcg:0.94340
[18]	validation_0-ndcg:0.94386
[20]	validation_0-ndcg:0.94531
[22]	validation_0-ndcg:0.94569
[24]	validation_0-ndcg:0.94516
[26]	validation_0-ndcg:0.94538
[28]	validation_0-ndcg:0.94572
[30]	validation_0-ndcg:0.94650
[32]	validation_0-ndcg:0.94625
[34]	validation_0-ndcg:0.94593
[36]	validation_0-ndcg:0.94548
[38]	validation_0-ndcg:0.94568
[40]	validation_0-ndcg:0.94638
[42]	validation_0-ndcg:0.94686
[44]	validation_0-ndcg:0.94696
[46]	validation_0-ndcg:0.94713
[48]	validation_0-ndcg:0.94666
[50]	validation_0-ndcg:0.94620
[52]	validation_0-ndcg:0.94698
[54]	validation_0-ndcg:0.94672
[56]	validation_0-ndcg:0.94691
[58]	validation_0-ndcg:0.94685
[60]	validation_0-ndcg:0.94672
[62]	validation_0-ndcg:0.94690
[64]	validati

In [90]:
best_param = study.best_params
print(study.best_params)
print(study.best_value)

{'n_estimators': 96, 'learning_rate': 0.014327766729298533, 'max_depth': 5, 'max_leaves': 11, 'reg_alpha': 0.0035401431789771516, 'reg_lambda': 0.004520715924780918, 'subsample': 0.7684184054295675, 'colsample_bytree': 0.9318761708581184, 'min_child_weight': 4, 'gamma': 1.6432477526172597}
0.9540804791921766


In [94]:
fig = optuna.visualization.plot_optimization_history(study)
show.show(fig)

In [ ]:

ranker = XGBRanker(tree_method = 'hist', objective="rank:ndcg", learning_rate=0.05, max_depth=6, n_estimators=100 ,ndcg_exp_gain=False)
ranker.fit(X_train, y_train,qid=qid_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [ ]:
print(ranker.feature_importances_)
print(y_train.value_counts())
print(qid_train.dtypes)
print(pd.Series(qid_train).value_counts().head())

[0.04042294 0.04984919 0.04329793 0.3063513  0.38647905 0.05941573
 0.06848595 0.04569793]
relevance_score
22.0    125
3.0     125
15.0    125
16.0    125
21.0    125
18.0    125
19.0    125
20.0    125
13.0    125
14.0    125
8.0     125
12.0    125
11.0    125
10.0    125
9.0     125
7.0     125
6.0     125
5.0     125
4.0     125
17.0    125
Name: count, dtype: int64
int64
session_key
5066    20
6714    20
7128    20
7115    20
7105    20
Name: count, dtype: int64


In [ ]:
cols_eval = ['air_temp', 'rainfall', 'wind_speed', 'quali_relative_time', 'driver_points', 'constructor_points', 'constructors_ewma', 'driver_ewma']
one_session = test_df[one_session_mask].copy()
one_session.info()
one_session.insert(0,'prediction',ranker.predict(one_session[cols_eval]))
one_session.sort_values(by='prediction', ascending=False)

<class 'pandas.core.frame.DataFrame'>
Index: 22 entries, 3735 to 3732
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   year                 22 non-null     int64  
 1   session_key          22 non-null     int64  
 2   driver_id            22 non-null     object 
 3   air_temp             22 non-null     float64
 4   rainfall             22 non-null     float64
 5   wind_speed           22 non-null     float64
 6   quali_relative_time  22 non-null     float64
 7   driver_points        22 non-null     float64
 8   constructor_points   22 non-null     float64
 9   constructors_ewma    22 non-null     float64
 10  driver_ewma          22 non-null     float64
 11  relevance_score      22 non-null     float64
dtypes: float64(9), int64(2), object(1)
memory usage: 2.2+ KB


,prediction,year,session_key,driver_id,air_temp,rainfall,wind_speed,quali_relative_time,driver_points,constructor_points,constructors_ewma,driver_ewma,relevance_score
3722,2.109710,2026,11361,antonelli,32.172727,0.0,1.347159,96.0,267.0,468.0,8.914423,2.901710,22.0
3727,1.065528,2026,11361,hamilton,32.172727,0.0,1.347159,225.0,191.0,346.0,8.623929,4.182677,17.0
3743,0.982414,2026,11361,leclerc,32.172727,0.0,1.347159,218.0,155.0,346.0,8.623929,4.441252,1.0
3728,0.815895,2026,11361,gasly,32.172727,0.0,1.347159,0.0,41.0,62.0,24.166822,10.628091,16.0
3723,0.771289,2026,11361,russell,32.172727,0.0,1.347159,60.0,201.0,468.0,8.914423,6.012713,21.0
3725,0.724279,2026,11361,norris,32.172727,0.0,1.347159,281.0,171.0,287.0,11.575531,2.047086,19.0
3724,0.716789,2026,11361,max_verstappen,32.172727,0.0,1.347159,284.0,127.0,204.0,20.789975,14.275392,20.0
3726,0.659158,2026,11361,piastri,32.172727,0.0,1.347159,180.0,116.0,287.0,11.575531,9.528445,18.0
3735,0.269555,2026,11361,lawson,32.172727,0.0,1.347159,1035.0,51.0,204.0,20.789975,7.795594,9.0
3729,0.248204,2026,11361,arvid_lindblad,32.172727,0.0,1.347159,500.0,29.0,75.0,20.905025,10.868616,15.0
